# Fleet Allocation Optimization

## Public Transport Demand Planning — Kharagpur → Kolkata Corridor

This notebook converts predicted passenger demand into an operational
fleet allocation plan using mathematical optimization.

### Decision Objective

Determine the number of bus and rail services required on each corridor
segment while minimizing:

- Operating cost
- Unmet passenger demand

subject to fleet and service-capacity constraints.

In [1]:
import pandas as pd
import numpy as np
import pulp

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# ============================================
# 3. LOAD DEMAND FORECAST
# ============================================

forecast_output = pd.read_csv(
    "../outputs/demand_forecast.csv"
)

forecast_output["date"] = pd.to_datetime(
    forecast_output["date"]
)

print("Forecast data loaded successfully.")
print("Shape:", forecast_output.shape)
print("\nDate range:")
print(
    forecast_output["date"].min(),
    "to",
    forecast_output["date"].max()
)

print("\nColumns:")
print(forecast_output.columns.tolist())

forecast_output.head()

Forecast data loaded successfully.
Shape: (128, 5)

Date range:
2025-10-14 00:00:00 to 2025-10-29 00:00:00

Columns:
['date', 'route', 'passengers', 'predicted_demand', 'forecast_error']


,date,route,passengers,predicted_demand,forecast_error
0,2025-10-14,Howrah_Kolkata_Bus,6794,6227,567
1,2025-10-15,Howrah_Kolkata_Bus,5529,6683,-1154
2,2025-10-16,Howrah_Kolkata_Bus,6601,6513,88
3,2025-10-17,Howrah_Kolkata_Bus,7549,6657,892
4,2025-10-18,Howrah_Kolkata_Bus,6539,5557,982


In [3]:
# ============================================
# 4. OPERATING ASSUMPTIONS
# ============================================

BUS_CAPACITY = 50
RAIL_CAPACITY = 1000

BUS_COST = 1
RAIL_COST = 8

MAX_BUS_SERVICES = 400
MAX_RAIL_SERVICES = 80

UNMET_DEMAND_PENALTY = 20

print("Operating assumptions loaded.")
print(f"Bus capacity       : {BUS_CAPACITY}")
print(f"Rail capacity      : {RAIL_CAPACITY}")
print(f"Bus cost/service   : {BUS_COST}")
print(f"Rail cost/service  : {RAIL_COST}")
print(f"Max bus services   : {MAX_BUS_SERVICES}")
print(f"Max rail services  : {MAX_RAIL_SERVICES}")
print(f"Unmet penalty      : {UNMET_DEMAND_PENALTY}")

Operating assumptions loaded.
Bus capacity       : 50
Rail capacity      : 1000
Bus cost/service   : 1
Rail cost/service  : 8
Max bus services   : 400
Max rail services  : 80
Unmet penalty      : 20


In [4]:
# ============================================
# 5. FLEET AVAILABILITY
# ============================================

TOTAL_BUS_FLEET = 40
TOTAL_RAIL_FLEET = 50

print("Fleet availability:")
print(f"Total buses available per day : {TOTAL_BUS_FLEET}")
print(f"Total rail services available : {TOTAL_RAIL_FLEET}")

Fleet availability:
Total buses available per day : 40
Total rail services available : 50


In [5]:
# ============================================
# 6. SELECT PLANNING DAY
# ============================================

planning_date = forecast_output["date"].min()

daily_forecast = forecast_output[
    forecast_output["date"] == planning_date
].copy()

print("Planning date:", planning_date.date())
print("\nForecast demand:")
print(
    daily_forecast[
        ["route", "predicted_demand"]
    ].to_string(index=False)
)

print(
    "\nTotal forecast demand:",
    round(daily_forecast["predicted_demand"].sum())
)

Planning date: 2025-10-14

Forecast demand:
                   route  predicted_demand
      Howrah_Kolkata_Bus              6227
     Howrah_Kolkata_Rail             14524
 Kharagpur_Midnapore_Bus              8462
Kharagpur_Midnapore_Rail             11205
  Midnapore_Uluberia_Bus              6568
 Midnapore_Uluberia_Rail             14519
     Uluberia_Howrah_Bus              6623
    Uluberia_Howrah_Rail             12661

Total forecast demand: 80789


In [6]:
# ============================================
# 7. FLEET-CONSTRAINED OPTIMIZATION MODEL
# ============================================

model = pulp.LpProblem(
    "Fleet_Allocation_Optimization",
    pulp.LpMinimize
)

routes = daily_forecast["route"].tolist()

# Decision variables
bus_services = {
    route: pulp.LpVariable(
        f"bus_{route}",
        lowBound=0,
        cat="Integer"
    )
    for route in routes
}

rail_services = {
    route: pulp.LpVariable(
        f"rail_{route}",
        lowBound=0,
        cat="Integer"
    )
    for route in routes
}

unmet_demand = {
    route: pulp.LpVariable(
        f"unmet_{route}",
        lowBound=0,
        cat="Continuous"
    )
    for route in routes
}

# --------------------------------------------
# Objective
# --------------------------------------------

model += pulp.lpSum(
    BUS_COST * bus_services[r]
    + RAIL_COST * rail_services[r]
    + UNMET_DEMAND_PENALTY * unmet_demand[r]
    for r in routes
)

# --------------------------------------------
# Route-level demand constraints
# --------------------------------------------

for _, row in daily_forecast.iterrows():

    route = row["route"]
    demand = row["predicted_demand"]

    model += (
        BUS_CAPACITY * bus_services[route]
        + RAIL_CAPACITY * rail_services[route]
        + unmet_demand[route]
        >= demand
    )

# --------------------------------------------
# Fleet-wide constraints
# --------------------------------------------

model += pulp.lpSum(
    bus_services[r] for r in routes
) <= TOTAL_BUS_FLEET

model += pulp.lpSum(
    rail_services[r] for r in routes
) <= TOTAL_RAIL_FLEET

# --------------------------------------------
# Solve
# --------------------------------------------

model.solve(
    pulp.PULP_CBC_CMD(msg=False)
)

print("Optimization status:")
print(pulp.LpStatus[model.status])

Optimization status:
Optimal


In [9]:
# ============================================
# 8. EXTRACT OPTIMAL FLEET ALLOCATION
# ============================================

optimization_results = []

for route in routes:

    demand = daily_forecast.loc[
        daily_forecast["route"] == route,
        "predicted_demand"
    ].iloc[0]

    buses = int(round(pulp.value(bus_services[route])))
    rail = int(round(pulp.value(rail_services[route])))
    unmet = pulp.value(unmet_demand[route])

    capacity = (
        BUS_CAPACITY * buses
        + RAIL_CAPACITY * rail
    )

    optimization_results.append({
        "route": route,
        "forecast_demand": round(demand),
        "bus_services": buses,
        "rail_services": rail,
        "capacity_provided": capacity,
        "unmet_demand": round(unmet, 2),
        "capacity_coverage_pct": round(
    min(demand, capacity) / demand * 100,
    2
) if demand > 0 else 0
    })

optimization_results = pd.DataFrame(
    optimization_results
)

optimization_results

,route,forecast_demand,bus_services,rail_services,capacity_provided,unmet_demand,capacity_coverage_pct
0,Howrah_Kolkata_Bus,6227,4,6,6200,27.0,99.57
1,Howrah_Kolkata_Rail,14524,2,0,100,14424.0,0.69
2,Kharagpur_Midnapore_Bus,8462,0,0,0,8462.0,0.00
3,Kharagpur_Midnapore_Rail,11205,0,6,6000,5205.0,53.55
4,Midnapore_Uluberia_Bus,6568,11,6,6550,18.0,99.73
5,Midnapore_Uluberia_Rail,14519,10,14,14500,19.0,99.87
6,Uluberia_Howrah_Bus,6623,0,6,6000,623.0,90.59
7,Uluberia_Howrah_Rail,12661,13,12,12650,11.0,99.91


In [10]:
# ============================================
# NETWORK CAPACITY GAP
# ============================================

total_forecast_demand = (
    optimization_results["forecast_demand"].sum()
)

total_capacity = (
    optimization_results["capacity_provided"].sum()
)

total_unmet = (
    optimization_results["unmet_demand"].sum()
)

coverage = (
    (total_forecast_demand - total_unmet)
    / total_forecast_demand
    * 100
)

print("NETWORK CAPACITY ANALYSIS")
print("-----------------------------------")
print(f"Forecast demand       : {total_forecast_demand:,}")
print(f"Capacity provided     : {total_capacity:,}")
print(f"Unmet demand          : {total_unmet:,.0f}")
print(f"Demand coverage       : {coverage:.2f}%")
print(
    f"Capacity gap          : "
    f"{total_forecast_demand - total_capacity:,}"
)

NETWORK CAPACITY ANALYSIS
-----------------------------------
Forecast demand       : 80,789
Capacity provided     : 52,000
Unmet demand          : 28,789
Demand coverage       : 64.37%
Capacity gap          : 28,789


In [8]:
# ============================================
# 9. NETWORK-LEVEL SUMMARY
# ============================================

total_buses_used = optimization_results["bus_services"].sum()
total_rail_used = optimization_results["rail_services"].sum()
total_unmet = optimization_results["unmet_demand"].sum()

total_cost = (
    total_buses_used * BUS_COST
    + total_rail_used * RAIL_COST
    + total_unmet * UNMET_DEMAND_PENALTY
)

print("NETWORK OPTIMIZATION SUMMARY")
print("-----------------------------------")
print(f"Buses used        : {total_buses_used}")
print(f"Buses available   : {TOTAL_BUS_FLEET}")
print(f"Rail services used: {total_rail_used}")
print(f"Rail available    : {TOTAL_RAIL_FLEET}")
print(f"Total unmet demand: {total_unmet:.2f}")
print(f"Total operating cost: {total_cost:.2f}")

NETWORK OPTIMIZATION SUMMARY
-----------------------------------
Buses used        : 40
Buses available   : 40
Rail services used: 50
Rail available    : 50
Total unmet demand: 28789.00
Total operating cost: 576220.00


## 10. Corrected Multimodal Fleet Allocation Model

The initial optimization formulation treated each route-mode combination
as an independent route. This could allow a vehicle type to be assigned
to a route labelled for another mode.

To represent the transportation system more realistically, the corrected
model separates:

- Physical corridor segment
- Rail demand
- Bus demand
- Rail fleet allocation
- Bus fleet allocation

The optimization therefore assigns buses only to bus demand and rail
services only to rail demand, while both modes compete for their respective
limited fleet resources.

In [11]:
# ============================================
# 10. CREATE PHYSICAL CORRIDOR SEGMENTS
# ============================================

# Extract origin and destination from route
forecast_output[["origin", "destination", "mode"]] = (
    forecast_output["route"].str.split("_", expand=True)
)

forecast_output["segment"] = (
    forecast_output["origin"]
    + " → "
    + forecast_output["destination"]
)

# Select the same planning day
daily_forecast = forecast_output[
    forecast_output["date"] == planning_date
].copy()

print("Physical corridor segments:")
print(
    daily_forecast["segment"]
    .unique()
)

Physical corridor segments:
['Howrah → Kolkata' 'Kharagpur → Midnapore' 'Midnapore → Uluberia'
 'Uluberia → Howrah']


In [12]:
# ============================================
# 11. MODE-SPECIFIC DEMAND
# ============================================

segment_demand = (
    daily_forecast
    .pivot_table(
        index="segment",
        columns="mode",
        values="predicted_demand",
        aggfunc="sum"
    )
    .fillna(0)
    .reset_index()
)

# Make sure both columns exist
if "Bus" not in segment_demand.columns:
    segment_demand["Bus"] = 0

if "Rail" not in segment_demand.columns:
    segment_demand["Rail"] = 0

segment_demand = segment_demand[
    ["segment", "Bus", "Rail"]
]

segment_demand.columns = [
    "segment",
    "bus_demand",
    "rail_demand"
]

segment_demand

,segment,bus_demand,rail_demand
0,Howrah → Kolkata,6227,14524
1,Kharagpur → Midnapore,8462,11205
2,Midnapore → Uluberia,6568,14519
3,Uluberia → Howrah,6623,12661


In [13]:
# ============================================
# 12. CORRECTED FLEET OPTIMIZATION
# ============================================

corrected_model = pulp.LpProblem(
    "Corrected_Multimodal_Fleet_Allocation",
    pulp.LpMinimize
)

segments = segment_demand["segment"].tolist()

# --------------------------------------------
# Decision variables
# --------------------------------------------

bus_services_corrected = {
    segment: pulp.LpVariable(
        f"bus_{segment}",
        lowBound=0,
        cat="Integer"
    )
    for segment in segments
}

rail_services_corrected = {
    segment: pulp.LpVariable(
        f"rail_{segment}",
        lowBound=0,
        cat="Integer"
    )
    for segment in segments
}

unmet_bus = {
    segment: pulp.LpVariable(
        f"unmet_bus_{segment}",
        lowBound=0,
        cat="Continuous"
    )
    for segment in segments
}

unmet_rail = {
    segment: pulp.LpVariable(
        f"unmet_rail_{segment}",
        lowBound=0,
        cat="Continuous"
    )
    for segment in segments
}

In [14]:
# ============================================
# OBJECTIVE FUNCTION
# ============================================

corrected_model += pulp.lpSum(
    BUS_COST * bus_services_corrected[s]
    + RAIL_COST * rail_services_corrected[s]
    + UNMET_DEMAND_PENALTY * (
        unmet_bus[s] + unmet_rail[s]
    )
    for s in segments
)

In [15]:
# ============================================
# MODE-SPECIFIC DEMAND CONSTRAINTS
# ============================================

for _, row in segment_demand.iterrows():

    segment = row["segment"]

    bus_demand = row["bus_demand"]
    rail_demand = row["rail_demand"]

    # Bus demand can only be served by buses
    corrected_model += (
        BUS_CAPACITY * bus_services_corrected[segment]
        + unmet_bus[segment]
        >= bus_demand
    )

    # Rail demand can only be served by rail services
    corrected_model += (
        RAIL_CAPACITY * rail_services_corrected[segment]
        + unmet_rail[segment]
        >= rail_demand
    )

In [16]:
# ============================================
# FLEET-WIDE RESOURCE CONSTRAINTS
# ============================================

corrected_model += pulp.lpSum(
    bus_services_corrected[s]
    for s in segments
) <= TOTAL_BUS_FLEET

corrected_model += pulp.lpSum(
    rail_services_corrected[s]
    for s in segments
) <= TOTAL_RAIL_FLEET

In [17]:
# ============================================
# SOLVE CORRECTED MODEL
# ============================================

corrected_model.solve(
    pulp.PULP_CBC_CMD(msg=False)
)

print(
    "Corrected optimization status:",
    pulp.LpStatus[corrected_model.status]
)

Corrected optimization status: Optimal


In [18]:
# ============================================
# 13. CORRECTED OPTIMIZATION RESULTS
# ============================================

corrected_results = []

for _, row in segment_demand.iterrows():

    segment = row["segment"]

    bus_demand = row["bus_demand"]
    rail_demand = row["rail_demand"]

    buses = int(
        round(
            pulp.value(
                bus_services_corrected[segment]
            )
        )
    )

    rail = int(
        round(
            pulp.value(
                rail_services_corrected[segment]
            )
        )
    )

    bus_unmet = pulp.value(
        unmet_bus[segment]
    )

    rail_unmet = pulp.value(
        unmet_rail[segment]
    )

    bus_capacity = buses * BUS_CAPACITY
    rail_capacity = rail * RAIL_CAPACITY

    corrected_results.append({
        "segment": segment,
        "bus_demand": round(bus_demand),
        "rail_demand": round(rail_demand),
        "bus_services": buses,
        "rail_services": rail,
        "bus_capacity": bus_capacity,
        "rail_capacity": rail_capacity,
        "bus_unmet": round(bus_unmet, 2),
        "rail_unmet": round(rail_unmet, 2)
    })

corrected_results = pd.DataFrame(
    corrected_results
)

corrected_results

,segment,bus_demand,rail_demand,bus_services,rail_services,bus_capacity,rail_capacity,bus_unmet,rail_unmet
0,Howrah → Kolkata,6227,14524,0,14,0,14000,6227.0,524.0
1,Kharagpur → Midnapore,8462,11205,40,11,2000,11000,6462.0,205.0
2,Midnapore → Uluberia,6568,14519,0,14,0,14000,6568.0,519.0
3,Uluberia → Howrah,6623,12661,0,11,0,11000,6623.0,1661.0


In [19]:
# ============================================
# 14. CORRECTED NETWORK SUMMARY
# ============================================

total_bus_demand = corrected_results["bus_demand"].sum()
total_rail_demand = corrected_results["rail_demand"].sum()

total_bus_capacity = corrected_results["bus_capacity"].sum()
total_rail_capacity = corrected_results["rail_capacity"].sum()

total_bus_unmet = corrected_results["bus_unmet"].sum()
total_rail_unmet = corrected_results["rail_unmet"].sum()

total_demand = total_bus_demand + total_rail_demand
total_capacity = total_bus_capacity + total_rail_capacity
total_unmet = total_bus_unmet + total_rail_unmet

demand_coverage = (
    (total_demand - total_unmet)
    / total_demand
    * 100
)

print("CORRECTED NETWORK SUMMARY")
print("-----------------------------------")
print(f"Total bus demand       : {total_bus_demand:,.0f}")
print(f"Total rail demand      : {total_rail_demand:,.0f}")
print(f"Total forecast demand  : {total_demand:,.0f}")
print()
print(f"Bus capacity provided  : {total_bus_capacity:,.0f}")
print(f"Rail capacity provided : {total_rail_capacity:,.0f}")
print(f"Total capacity         : {total_capacity:,.0f}")
print()
print(f"Bus unmet demand       : {total_bus_unmet:,.0f}")
print(f"Rail unmet demand      : {total_rail_unmet:,.0f}")
print(f"Total unmet demand     : {total_unmet:,.0f}")
print()
print(f"Demand coverage        : {demand_coverage:.2f}%")
print()
print(f"Bus services used      : {corrected_results['bus_services'].sum()}")
print(f"Rail services used     : {corrected_results['rail_services'].sum()}")

CORRECTED NETWORK SUMMARY
-----------------------------------
Total bus demand       : 27,880
Total rail demand      : 52,909
Total forecast demand  : 80,789

Bus capacity provided  : 2,000
Rail capacity provided : 50,000
Total capacity         : 52,000

Bus unmet demand       : 25,880
Rail unmet demand      : 2,909
Total unmet demand     : 28,789

Demand coverage        : 64.37%

Bus services used      : 40
Rail services used     : 50


In [20]:
# ============================================
# 15. REUSABLE FLEET OPTIMIZATION FUNCTION
# ============================================

def optimize_fleet(
    demand_data,
    total_buses,
    total_rail
):
    """
    Optimize multimodal fleet allocation for one planning day.

    Bus demand can only be served by buses.
    Rail demand can only be served by rail services.
    """

    model = pulp.LpProblem(
        "Fleet_Allocation",
        pulp.LpMinimize
    )

    segments = demand_data["segment"].tolist()

    # Decision variables
    bus_services = {
        s: pulp.LpVariable(
            f"bus_{s}",
            lowBound=0,
            cat="Integer"
        )
        for s in segments
    }

    rail_services = {
        s: pulp.LpVariable(
            f"rail_{s}",
            lowBound=0,
            cat="Integer"
        )
        for s in segments
    }

    unmet_bus = {
        s: pulp.LpVariable(
            f"unmet_bus_{s}",
            lowBound=0
        )
        for s in segments
    }

    unmet_rail = {
        s: pulp.LpVariable(
            f"unmet_rail_{s}",
            lowBound=0
        )
        for s in segments
    }

    # Objective
    model += pulp.lpSum(
        BUS_COST * bus_services[s]
        + RAIL_COST * rail_services[s]
        + UNMET_DEMAND_PENALTY *
        (unmet_bus[s] + unmet_rail[s])
        for s in segments
    )

    # Demand constraints
    for _, row in demand_data.iterrows():

        s = row["segment"]

        model += (
            BUS_CAPACITY * bus_services[s]
            + unmet_bus[s]
            >= row["bus_demand"]
        )

        model += (
            RAIL_CAPACITY * rail_services[s]
            + unmet_rail[s]
            >= row["rail_demand"]
        )

    # Fleet constraints
    model += pulp.lpSum(
        bus_services[s]
        for s in segments
    ) <= total_buses

    model += pulp.lpSum(
        rail_services[s]
        for s in segments
    ) <= total_rail

    # Solve
    model.solve(
        pulp.PULP_CBC_CMD(msg=False)
    )

    # Results
    total_unmet = sum(
        pulp.value(unmet_bus[s])
        + pulp.value(unmet_rail[s])
        for s in segments
    )

    total_bus_used = sum(
        pulp.value(bus_services[s])
        for s in segments
    )

    total_rail_used = sum(
        pulp.value(rail_services[s])
        for s in segments
    )

    total_capacity = (
        total_bus_used * BUS_CAPACITY
        + total_rail_used * RAIL_CAPACITY
    )

    total_demand = (
        demand_data["bus_demand"].sum()
        + demand_data["rail_demand"].sum()
    )

    coverage = (
        (total_demand - total_unmet)
        / total_demand
        * 100
    )

    operating_cost = (
        total_bus_used * BUS_COST
        + total_rail_used * RAIL_COST
    )

    return {
        "buses_available": total_buses,
        "rail_available": total_rail,
        "buses_used": round(total_bus_used),
        "rail_used": round(total_rail_used),
        "capacity": round(total_capacity),
        "unmet_demand": round(total_unmet),
        "coverage_pct": round(coverage, 2),
        "operating_cost": round(operating_cost, 2)
    }

In [21]:
# ============================================
# 16. BASELINE SCENARIO
# ============================================

baseline = optimize_fleet(
    segment_demand,
    total_buses=40,
    total_rail=50
)

baseline

{'buses_available': 40,
 'rail_available': 50,
 'buses_used': 40,
 'rail_used': 50,
 'capacity': 52000,
 'unmet_demand': 28789,
 'coverage_pct': np.float64(64.37),
 'operating_cost': 440.0}

In [22]:
# ============================================
# 17. FLEET EXPANSION SCENARIOS
# ============================================

scenarios = [
    {
        "scenario": "Current Fleet",
        "buses": 40,
        "rail": 50
    },
    {
        "scenario": "+10 Buses",
        "buses": 50,
        "rail": 50
    },
    {
        "scenario": "+10 Rail",
        "buses": 40,
        "rail": 60
    },
    {
        "scenario": "+20 Buses",
        "buses": 60,
        "rail": 50
    },
    {
        "scenario": "+20 Rail",
        "buses": 40,
        "rail": 70
    },
    {
        "scenario": "+10 Bus +10 Rail",
        "buses": 50,
        "rail": 60
    }
]

scenario_results = []

for scenario in scenarios:

    result = optimize_fleet(
        segment_demand,
        total_buses=scenario["buses"],
        total_rail=scenario["rail"]
    )

    result["scenario"] = scenario["scenario"]

    scenario_results.append(result)

scenario_results = pd.DataFrame(
    scenario_results
)

scenario_results = scenario_results[
    [
        "scenario",
        "buses_available",
        "rail_available",
        "buses_used",
        "rail_used",
        "capacity",
        "unmet_demand",
        "coverage_pct",
        "operating_cost"
    ]
]

scenario_results

,scenario,buses_available,rail_available,buses_used,rail_used,capacity,unmet_demand,coverage_pct,operating_cost
0,Current Fleet,40,50,40,50,52000,28789,64.37,440.0
1,+10 Buses,50,50,50,50,52500,28289,64.98,450.0
2,+10 Rail,40,60,40,55,57000,25880,67.97,480.0
3,+20 Buses,60,50,60,50,53000,27789,65.60,460.0
4,+20 Rail,40,70,40,55,57000,25880,67.97,480.0
5,+10 Bus +10 Rail,50,60,50,55,57500,25380,68.58,490.0


In [23]:
# ============================================
# 18. OPTIMIZATION ACROSS ALL TEST DAYS
# ============================================

daily_results = []

for date in sorted(forecast_output["date"].unique()):

    day_data = forecast_output[
        forecast_output["date"] == date
    ].copy()

    # Convert to segment-level mode demand
    day_segment_demand = (
        day_data
        .pivot_table(
            index="segment",
            columns="mode",
            values="predicted_demand",
            aggfunc="sum"
        )
        .fillna(0)
        .reset_index()
    )

    if "Bus" not in day_segment_demand.columns:
        day_segment_demand["Bus"] = 0

    if "Rail" not in day_segment_demand.columns:
        day_segment_demand["Rail"] = 0

    day_segment_demand = day_segment_demand[
        ["segment", "Bus", "Rail"]
    ]

    day_segment_demand.columns = [
        "segment",
        "bus_demand",
        "rail_demand"
    ]

    result = optimize_fleet(
        day_segment_demand,
        total_buses=40,
        total_rail=50
    )

    result["date"] = date

    daily_results.append(result)

daily_results = pd.DataFrame(daily_results)

daily_results = daily_results[
    [
        "date",
        "buses_available",
        "rail_available",
        "buses_used",
        "rail_used",
        "capacity",
        "unmet_demand",
        "coverage_pct",
        "operating_cost"
    ]
]

daily_results

,date,buses_available,rail_available,buses_used,rail_used,capacity,unmet_demand,coverage_pct,operating_cost
0,2025-10-14,40,50,40,50,52000,28789,64.37,440.0
1,2025-10-15,40,50,40,50,52000,28384,64.69,440.0
2,2025-10-16,40,50,40,50,52000,30027,63.39,440.0
3,2025-10-17,40,50,40,50,52000,28833,64.33,440.0
4,2025-10-18,40,50,40,48,50000,24005,66.46,424.0
5,2025-10-19,40,50,40,46,48000,23371,66.28,408.0
6,2025-10-20,40,50,40,50,52000,98864,34.47,440.0
7,2025-10-21,40,50,40,50,52000,30704,62.87,440.0
8,2025-10-22,40,50,40,50,52000,33025,61.16,440.0
9,2025-10-23,40,50,40,50,52000,31087,62.59,440.0


In [24]:
# ============================================
# 19. INVESTIGATE EXTREME DEMAND DAY
# ============================================

extreme_date = daily_results.loc[
    daily_results["unmet_demand"].idxmax(),
    "date"
]

print("Extreme demand date:", extreme_date)

extreme_day = forecast_output[
    forecast_output["date"] == extreme_date
].copy()

print("\nRoute-level demand:")
print(
    extreme_day[
        [
            "route",
            "passengers",
            "predicted_demand",
            "forecast_error"
        ]
    ].to_string(index=False)
)

print("\nTotal actual demand:",
      extreme_day["passengers"].sum())

print("Total predicted demand:",
      extreme_day["predicted_demand"].sum())

Extreme demand date: 2025-10-20 00:00:00

Route-level demand:
                   route  passengers  predicted_demand  forecast_error
      Howrah_Kolkata_Bus       11624             11867            -243
     Howrah_Kolkata_Rail       23359             26119           -2760
 Kharagpur_Midnapore_Bus       17284             15101            2183
Kharagpur_Midnapore_Rail       21237             22259           -1022
  Midnapore_Uluberia_Bus        9590             12117           -2527
 Midnapore_Uluberia_Rail       24873             24846              27
     Uluberia_Howrah_Bus       13929             13796             133
    Uluberia_Howrah_Rail       23112             24759           -1647

Total actual demand: 145008
Total predicted demand: 150864


In [27]:
forecast_output = pd.read_csv(
    "../outputs/demand_forecast.csv"
)

In [28]:
forecast_output.columns.tolist()

['date',
 'route',
 'passengers',
 'festival',
 'weekend',
 'day_of_week',
 'predicted_demand',
 'forecast_error']

In [30]:
# ============================================
# 20. FESTIVAL STATUS OF EXTREME DAY
# ============================================

# Make sure dates use the same format
forecast_output["date"] = pd.to_datetime(forecast_output["date"])
extreme_date = pd.to_datetime(extreme_date)

print("Extreme date:", extreme_date)

festival_check = (
    forecast_output[
        forecast_output["date"] == extreme_date
    ][
        ["date", "festival"]
    ]
    .drop_duplicates()
)

festival_check

Extreme date: 2025-10-20 00:00:00


,date,festival
6,2025-10-20,1


In [31]:
# ============================================
# 21. EXTREME DAY DEMAND ANALYSIS
# ============================================

extreme_day = (
    forecast_output[
        forecast_output["date"] == extreme_date
    ][
        [
            "date",
            "route",
            "passengers",
            "predicted_demand",
            "festival",
            "weekend",
            "forecast_error"
        ]
    ]
    .copy()
)

print("EXTREME DAY:", extreme_date)
print("\nRoute-level demand:")
display(extreme_day)

print("\nTotal actual demand:",
      extreme_day["passengers"].sum())

print("Total predicted demand:",
      extreme_day["predicted_demand"].sum())

print("Festival status:",
      extreme_day["festival"].iloc[0])

print("Weekend status:",
      extreme_day["weekend"].iloc[0])

EXTREME DAY: 2025-10-20 00:00:00

Route-level demand:


,date,route,passengers,predicted_demand,festival,weekend,forecast_error
6,2025-10-20,Howrah_Kolkata_Bus,11624,11867,1,0,-242.844887
22,2025-10-20,Howrah_Kolkata_Rail,23359,26119,1,0,-2760.390580
38,2025-10-20,Kharagpur_Midnapore_Bus,17284,15101,1,0,2182.936875
54,2025-10-20,Kharagpur_Midnapore_Rail,21237,22259,1,0,-1022.427135
70,2025-10-20,Midnapore_Uluberia_Bus,9590,12117,1,0,-2526.710981
86,2025-10-20,Midnapore_Uluberia_Rail,24873,24846,1,0,27.484310
102,2025-10-20,Uluberia_Howrah_Bus,13929,13796,1,0,132.683066
118,2025-10-20,Uluberia_Howrah_Rail,23112,24759,1,0,-1646.752230



Total actual demand: 145008
Total predicted demand: 150864
Festival status: 1
Weekend status: 0


In [32]:
extreme_day[
    [
        "route",
        "passengers",
        "predicted_demand",
        "festival",
        "forecast_error"
    ]
].sort_values(
    "predicted_demand",
    ascending=False
)

,route,passengers,predicted_demand,festival,forecast_error
22,Howrah_Kolkata_Rail,23359,26119,1,-2760.390580
86,Midnapore_Uluberia_Rail,24873,24846,1,27.484310
118,Uluberia_Howrah_Rail,23112,24759,1,-1646.752230
54,Kharagpur_Midnapore_Rail,21237,22259,1,-1022.427135
38,Kharagpur_Midnapore_Bus,17284,15101,1,2182.936875
102,Uluberia_Howrah_Bus,13929,13796,1,132.683066
70,Midnapore_Uluberia_Bus,9590,12117,1,-2526.710981
6,Howrah_Kolkata_Bus,11624,11867,1,-242.844887
